<a href="https://colab.research.google.com/github/Ramy99999999/Flyrank-machine-learning-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ramy99999999/Flyrank-machine-learning-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content item for one client (`client_hash_id × content_hash_id`), aggregated over a monthly window.

**Time window:** February 2026 is the feature window and March 2026 is the outcome/label window.

The windows do not overlap: features use information available by the end of February, while March contains the future outcome we want to predict.

The June 2026 `_sample` data is excluded from development because it is the final month and should be treated as a sealed test month.

In [36]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

In [37]:
import os
import duckdb
import pandas as pd

# Get the Hugging Face token from Colab Secrets
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Give DuckDB the Hugging Face token without putting it directly in a query
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# Warehouse locations
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# Time windows
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Outcome/label window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Outcome/label window: March 2026


In [38]:
con.execute(f"""
    SELECT *
    FROM {FEB}
    LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57,0,1778,...,0,0,0,0,0,0,0,0,0,2026-02
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,True,True,False,13,0,85,...,0,0,0,0,0,0,0,0,0,2026-02
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,True,True,True,59,0,1001,...,0,0,0,0,0,0,0,0,0,2026-02
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,True,True,False,17,0,287,...,0,0,0,0,0,0,0,0,0,2026-02
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,True,True,False,6,0,27,...,0,0,0,0,0,0,0,0,0,2026-02


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** February GSC impressions, February GSC clicks, February average search position, February organic sessions, and February scroll events.

**Label:** `went_dark = 1` when a content item has zero measured GSC clicks during March 2026. Content items with no measured March GSC days are excluded because missing measurement should not be treated as zero traffic.

**Context:** `client_hash_id`, `content_hash_id`, and the report month identify the content item and time window but are not predictive features.

**Excluded:** March performance fields and `trend_direction` are excluded because March information is only known after the February decision point, and `trend_direction` is not available in the warehouse fact table.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
universe_feb = con.execute(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id
FROM {FEB} AS f
JOIN read_parquet('{DIM_CONTENT}') AS d
    ON f.client_hash_id = d.client_hash_id
    AND f.content_hash_id = d.content_hash_id
GROUP BY
    f.client_hash_id,
    f.content_hash_id,
    d.is_published,
    d.content_created_date
HAVING
    SUM(f.gsc_impressions) >= 100
    AND SUM(f.gsc_clicks) >= 3
    AND d.is_published IS TRUE
    AND d.content_created_date <= DATE '2026-02-28'
""").df()

universe_feb.shape

(29700, 2)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Grain check:** After grouping the February data by client, content item, and month, there are 321,546 monthly rows and 321,546 unique monthly keys. The matching counts support the stated monthly grain.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.execute(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_TRUNC('month', report_date) AS month
    FROM {FEB}
    GROUP BY
        client_hash_id,
        content_hash_id,
        DATE_TRUNC('month', report_date)
)
SELECT
    COUNT(*) AS monthly_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || month) AS unique_monthly_keys
FROM monthly
""").df()

grain_check

,monthly_rows,unique_monthly_keys
0,321546,321546


In [41]:
window_check = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FEB}
""").df()

window_check

,row_count,first_date,last_date
0,7355108,2026-02-01,2026-02-28


In [42]:
availability_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM {FEB}
""").df()

availability_check

,total_rows,gsc_available_rows
0,7355108,2621783


**Five features:**

**1. February GSC impressions:** Available at the decision moment because they summarize search visibility observed during the February feature window.

**2. February GSC clicks:** Available at the decision moment because they summarize search clicks observed during the February feature window.

**3. February average search position:** Available at the decision moment because it is calculated from search-position data observed during the February feature window.

**4. February organic sessions:** Available at the decision moment because they summarize organic site sessions observed during the February feature window.

**5. February scroll events:** Available at the decision moment because they summarize engagement events observed during the February feature window.

In [43]:
features_feb = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions_feb,
    SUM(gsc_clicks) AS gsc_clicks_feb,
    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN gsc_sum_position
            ELSE 0
        END
    ) / NULLIF(
        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ),
        0
    ) AS avg_position_feb,
    SUM(sessions_organic) AS sessions_organic_feb,
    SUM(scroll_events) AS scroll_events_feb
FROM {FEB}
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

features_feb.head()

,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,avg_position_feb,sessions_organic_feb,scroll_events_feb
0,client_3ffa76342f366962,content_fb84747a57b8b665,0.0,0.0,NaN,NaN,NaN
1,client_3ffa76342f366962,content_feccf822ac21326e,0.0,0.0,NaN,NaN,NaN
2,client_3ffa76342f366962,content_17cf93c10413ebe9,0.0,0.0,NaN,NaN,NaN
3,client_3ffa76342f366962,content_a9905735266f8697,0.0,0.0,NaN,NaN,NaN
4,client_3ffa76342f366962,content_31c34765e7bba2f0,0.0,0.0,NaN,NaN,NaN


In [44]:
features_feb.shape

(321546, 7)

In [45]:
labels_mar = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS measured_days_mar,
    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_mar,
    CASE
        WHEN SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) = 0
        THEN 1
        ELSE 0
    END AS went_dark
FROM {MAR}
GROUP BY
    client_hash_id,
    content_hash_id
HAVING
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) > 0
""").df()

labels_mar.head()

,client_hash_id,content_hash_id,measured_days_mar,clicks_mar,went_dark
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,29,0.0,1
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,16,1.0,0
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,31,1.0,0
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,17,0.0,1
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,30,0.0,1


In [46]:
final_frame = (
    features_feb
    .merge(
        universe_feb,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
    .merge(
        labels_mar,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)

final_frame.shape

(29353, 10)

In [47]:
final_frame["went_dark"].value_counts()

,count
went_dark,
0,28194
1,1159


In [48]:
final_frame["went_dark"].mean()

np.float64(0.03948489081184206)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



### Leakage audit

A model should only use information available at the February decision point.

To demonstrate data leakage intentionally, I add a feature that directly copies the future `went_dark` label. This should produce an unrealistically high score because the model is effectively given the answer.

I then remove the leaking feature and compare the result using only the five February features.

In [49]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

feature_cols = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "avg_position_feb",
    "sessions_organic_feb",
    "scroll_events_feb"
]

model_frame = final_frame[
    [
        "client_hash_id",
        "content_hash_id",
        *feature_cols,
        "went_dark"
    ]
].copy()

# Deliberately create a leaking feature by copying the future label.
leaky = model_frame.copy()
leaky["leaked_label"] = leaky["went_dark"]

X_leaky = leaky[feature_cols + ["leaked_label"]]
y = leaky["went_dark"]

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leaky_model.fit(X_leaky, y)
leaky_pred = leaky_model.predict(X_leaky)

print(f"Leaky training accuracy: {accuracy_score(y, leaky_pred):.3f}")

Leaky training accuracy: 1.000


In [50]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
honest = model_frame.copy()

X_honest = honest[feature_cols]
y_honest = honest["went_dark"]

honest_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

honest_model.fit(X_honest, y_honest)
honest_pred = honest_model.predict(X_honest)

honest_accuracy = accuracy_score(y_honest, honest_pred)

print(f"Honest training accuracy: {honest_accuracy:.3f}")

Honest training accuracy: 0.961


**Leakage finding:** The deliberately leaky model achieved 1.000 training accuracy because `leaked_label` directly contained the future outcome. After removing that field, the honest model achieved 0.961 training accuracy using only the five February features.

This demonstrates why a very high model score can be misleading when future information is included. The five February features are available at the decision point, while the March outcome is not.

The honest accuracy is only a training check, not a final estimate of future performance. A proper evaluation would use a time-aware validation setup and a ranking metric such as Precision@K.

**Named limitation — unbalanced tracking history:** The warehouse contains different amounts of historical data and measurement coverage across clients. Therefore, missing data does not automatically mean that underlying activity was zero.

**GSC-only coverage:** Some rows have GSC measurements without equivalent GA4 measurements, so search visibility and site-engagement signals may not be equally available for every content item.

**Window limitation:** This contract uses February for features and March for the future outcome. It therefore supports a specific February-to-March prediction setup rather than proving that the same relationship will hold across every month.

**Leakage limitation:** March performance must not be used as a feature because it occurs after the February decision point.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.